In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from nltk.corpus import stopwords
import nltk
import sqlite3

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

db_path = '/Users/zphilipp/git/research/dealsdb/deals_db1.db'

prepositions_and_conjunctions = [
    "about", "above", "across", "after", "against", "along", "among", "around", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "to", "toward", "under",
    "until", "up", "with", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a", "b", "c", "d", "e", "f", "g", "h",
    "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w",
    "x", "y", "z"
]
pattern = r'\b(?:' + '|'.join(prepositions_and_conjunctions) + r')\b'

def remove_prepositions_and_conjunctions(text):
    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    return re.sub(r'\s+', ' ', cleaned_text).strip()

nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/zphilipp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

sql_query = """
    SELECT d.id, c.name || '. ' ||  GROUP_CONCAT(d.title_general) || '. '||GROUP_CONCAT (o.title) as text
    FROM deals d
        JOIN customer_category c ON c.id=d.customer_category_id
        JOIN options o ON o.deal_id=d.id
    GROUP BY d.id
"""

# Execute the query and load the data into a DataFrame
df = pd.read_sql_query(sql_query, conn)
conn.close()

In [5]:
df.head()

,id,text
0,1,"Barber Shop. Transform Your Look with Men's Haircut with Optional Wash and Beard Trim at 028 Barber School (Up to 64% Off),Transform Your Look with Men's Haircut with Optional Wash and Beard Trim at 028 Barber School (Up to 64% Off),Transform Your Look with Men's Haircut with Optional Wash and Beard Trim at 028 Barber School (Up to 64% Off). One Men's Haircut, Beard trim and Wash,One Men's Haircuts and Wash,One Men's Haircut"
1,2,"Things To Do. Experience Jersey Axe House with Admission for Groups of 4, 6, or 8 People and Save up to 22%,Experience Jersey Axe House with Admission for Groups of 4, 6, or 8 People and Save up to 22%. 2-Hour Axe Throwing Experience for 8 People (Ages 12 and Up),2-Hour Axe Throwing Experience for 6 People (Ages 12 and Up)"
2,3,"Junk Removal. Experience efficient junk removal with 1-800-Junk-Refund, hauling away 1/4 truck load of items, up to 56% off.. One Quarter Truck Load of Junk Removal"
3,4,Junk Removal. Clear Out Clutter With Our 1/4 Truck Load Junk Removal Service from 1-800-Junk-Refund (Up to 52% Off). 1/4 Truck Load of Junk Removal
4,5,"Bars. Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off,Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off,Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off,Explore the Country Bar Crawl with Las Vegas Tours featuring a Party Bus, Mixed Drinks, and VIP Access up to 58% off. Admit up to EIGHT: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks,Admit FOUR: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks,Admit TWO: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks,Admit ONE: #1 Country Bar Crawl in Vegas w/ Party Bus & Mixed Drinks"


In [6]:
tfidf_vectorizer = TfidfVectorizer(stop_words=stopwords.words('english'))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])

number_of_clusters = 100
kmeans = KMeans(n_clusters=number_of_clusters)
kmeans.fit(tfidf_matrix)


KMeans(n_clusters=100)

In [7]:
df['cluster'] = kmeans.labels_

In [14]:
df[['text', 'cluster']].query("cluster == 1").head(10)

,text,cluster
118,"Facial. One or Three Hydrodermabrasion Sessions at 2 Generations Beauty And Spa (Up to 50% Off),One or Three Hydrodermabrasion Sessions at 2 Generations Beauty And Spa (Up to 50% Off). One Hydrodermabrasion Session,Three Hydrodermabrasion Sessions",1
185,"Colonic Hydrotherapy. Up to 47% Off on Colonic / Hydro Colon Therapy at 247 Natural Wellness Center,Up to 47% Off on Colonic / Hydro Colon Therapy at 247 Natural Wellness Center. One Colon-Hydrotherapy (Colonic) Treatment for Returning Clients,One Colon-Hydrotherapy Treatment",1
251,"Natural Medicine. Experience advanced healing at 34th Street Chiropractic with varied Hyperbaric Oxygen Therapy sessions, offering up to 30% off. One 60-Minute Hyperbaric Oxygen Therapy Session",1
252,Spas. Discover 34th Street Chiropractic And Wellness' red light therapy sessions with up to 37% off. Red Light Therapy,1
255,"Weight Loss. Experience 360 Tan's Infrared Therapy Sessions for pain relief and skin health, offering up to 28% off without appointments.. Ten Infrared Red Light Therapy Sessions",1
263,"Facial. Experience rejuvenating facial oxygen therapy at 360 Radiance, offering advanced ultrasonic exfoliation and vacuum therapy up to 43% off. Oxygen Facial Therapy with Ultrasonic Exfoliation, Vacuum Therapy, and Customized Nutritional Serum",1
268,"Spas. Revitalize with One or Two Full Body Red Light Therapy Sessions at 360 Tans (Up to 74% Off),Revitalize with One or Two Full Body Red Light Therapy Sessions at 360 Tans (Up to 74% Off). One Full Body Red Light Therapy Session,Two Full Body Red Light Therapy Session",1
330,"Salt Caves. Experience the ultimate relaxation at 4 Elements Wellness Center with a 60-minute salt cave session for one or two, up to 46% off,Experience the ultimate relaxation at 4 Elements Wellness Center with a 60-minute salt cave session for one or two, up to 46% off. One 60-Minute Himalayan Salt Room Session for One,One 60-Minute Himalayan Salt Room Session for Two",1
336,"Medical. One or Three Energy & Metabolism Boosting Vitamin IV drip at 4Ever Young Doral(Up To 52% Off),One or Three Energy & Metabolism Boosting Vitamin IV drip at 4Ever Young Doral(Up To 52% Off). Three Energy & Metabolism Boosting Vitamin IV drip,One Energy & Metabolism Boosting Vitamin IV drip",1
338,"Medical. One or Three Energy & Metabolism Boosting Vitamin IV Drips at 4Ever Young Fleming Island (Up to 52% Off),One or Three Energy & Metabolism Boosting Vitamin IV Drips at 4Ever Young Fleming Island (Up to 52% Off). One Energy & Metabolism Boosting Vitamin IV Drip,Three Energy & Metabolism Boosting Vitamin IV Drip",1


In [15]:
df[['text', 'cluster']].query("cluster == 2").head(10)

,text,cluster
79,"Massage. Up to 51% Off on Lymphatic Drainage Massage at 124 Wellness Studio,Up to 51% Off on Lymphatic Drainage Massage at 124 Wellness Studio. Three 60-Minute Compression Lymphatic Drainage Massages for Legs & Massage,One 30-min Compression Lymphatic Drainage Massage For Legs Or Hips",2
103,"Weight Loss. One or Three Facial Endermologie w/ Instant Lift and Lymphatic Drainage at 1917 spa (Up to 60% Off),One or Three Facial Endermologie w/ Instant Lift and Lymphatic Drainage at 1917 spa (Up to 60% Off). 3 Facial Endermologie (instant lift and lymphatic drainage),1 Facial Endermologie (instant lift and lymphatic drainage)",2
569,"Massage. Get One or Three Lymphatic Drainage Sessions to Boost Wellness at 5 Elements Care and Solutions (Up To 60% Off),Get One or Three Lymphatic Drainage Sessions to Boost Wellness at 5 Elements Care and Solutions (Up To 60% Off). Three 45-Minute Lymphatic Drainage Sessions,One 45-Minute Lymphatic Drainage Session",2
739,"Massage. Lymphatic Drainage Therapy Sessions at A&N Beauty Bar (Up to 60% Off),Lymphatic Drainage Therapy Sessions at A&N Beauty Bar (Up to 60% Off). One Noninvasive Lymphatic Drainage Therapy Session,Three Noninvasive Lymphatic Drainage Therapy Sessions",2
1118,"Massage. 60-Minute Lymphatic Drainage Massage or Upgrade to 90-Minute Signature Option for Detoxification(Up To 55% Off),60-Minute Lymphatic Drainage Massage or Upgrade to 90-Minute Signature Option for Detoxification(Up To 55% Off). 60-Minute Lymphatic Drainage Massage,90-Minute Signature Lymphatic Drainage Massage",2
1445,"Massage. Experience tailored Brazilian lymphatic drainage massages at Abundance WellSpa with up to 54% off for enhanced recovery sessions.,Experience tailored Brazilian lymphatic drainage massages at Abundance WellSpa with up to 54% off for enhanced recovery sessions.,Experience tailored Brazilian lymphatic drainage massages at Abundance WellSpa with up to 54% off for enhanced recovery sessions.. One 45-Minute Brazilian Lymphatic Drainage Massage,6 45-Minute Brazilian Lymphatic Drainage Massage,Three 45-Minute Brazilian Lymphatic Drainage Massages",2
2115,"Massage. Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n,Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n,Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n,Discover Personalized Plans for Body Contouring with Lymphatic Drainage Treatments (Up to 90% Off)\n. One Fat Loss and Lymphatic Drainage Treatment,Four Fat Loss and Lymphatic Drainage Treatments,Three Fat Loss and Lymphatic Drainage Treatments,Two Fat Loss and Lymphatic Drainage Treatments",2
2192,Massage. Up to 37% Off on Lymphatic Drainage Massage at Aesthetically You and Weight Loss Too. 30-Minute Lymphatic Drainage Treatment,2
2441,"Massage. One 45-min Lymphatic Drainage Massage for Hips, Legs,/Arms (Compression) at Ageless Wellness Spa (Up to 37% Off),One 45-min Lymphatic Drainage Massage for Hips, Legs,/Arms (Compression) at Ageless Wellness Spa (Up to 37% Off),One 45-min Lymphatic Drainage Massage for Hips, Legs,/Arms (Compression) at Ageless Wellness Spa (Up to 37% Off). One 45-Minute Lymphatic Drainage Massage for arms (Compression),One 45-Minute Lymphatic Drainage Massage for hips (Compression),One 45-Minute Lymphatic Drainage Massage for legs (Compression)",2
2477,"Massage. Unwind at Ahoy Therapy LLC with Post-Op Lymphatic and Zero Gravity Massages up to 34% off,Unwind at Ahoy Therapy LLC with Post-Op Lymphatic and Zero Gravity Massages up to 34% off. Zero Gravity Chair Full Body Massage,Post Op Lymphatic Drainage Massage",2


In [13]:
df[['text', 'cluster']].query("cluster == 3").head(10)

,text,cluster
3270,"Mini Golf. One Round of Indoor Glow Golf for 2, 4, or 6 at Aloha Mini Glow Golf - Mall of New Hampshire (Up to 31% Off),One Round of Indoor Glow Golf for 2, 4, or 6 at Aloha Mini Glow Golf - Mall of New Hampshire (Up to 31% Off),One Round of Indoor Glow Golf for 2, 4, or 6 at Aloha Mini Glow Golf - Mall of New Hampshire (Up to 31% Off). One Round of Indoor Glow Golf for Six,One Round of Indoor Glow Golf for Two,One Round of Indoor Glow Golf for Four",3
11182,"Golf. Experience Custom Fit Clubs at Brians Golf Works with Options Worth $100, Up to 50% Off. $100 Golf Club Fitting Session",3
11343,"Golf. Enjoy a Relaxing 9-Hole Round of Golf for One, Two, or Four People at Brookland Golf Course (Up to 37% Off),Enjoy a Relaxing 9-Hole Round of Golf for One, Two, or Four People at Brookland Golf Course (Up to 37% Off),Enjoy a Relaxing 9-Hole Round of Golf for One, Two, or Four People at Brookland Golf Course (Up to 37% Off). 9-Hole Round of Golf for Two People,9-Hole Round of Golf for Four People,9-Hole Round of Golf for One Person",3
11727,"Golf. Tee Off Anytime with Up to Four Hours of Indoor Golf at Bunker Hill Golf Course (Up to 38% Off),Tee Off Anytime with Up to Four Hours of Indoor Golf at Bunker Hill Golf Course (Up to 38% Off),Tee Off Anytime with Up to Four Hours of Indoor Golf at Bunker Hill Golf Course (Up to 38% Off). Two Hours of Indoor Golf For Up to 6 People,One Hour of Indoor Golf For Up to 6 People,Four Hours of Indoor Golf For Up to 6 People",3
13881,"Golf. At Centerbrook Golf Course, experience 9 holes of golf with cart for up to 20% off,At Centerbrook Golf Course, experience 9 holes of golf with cart for up to 20% off. 9-Hole Round of Golf for Two with Cart,9-Hole Round of Golf for Four with Cart",3
14248,"Golf. 9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off),9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off),9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off),9- or 18-Round of Golf w/ Cart for 2 or 4 at Cherry Valley Golf Course(Up to 47% Off). 9-Hole Round of Golf with Cart for 4 (Monday-Friday Anytime, Sat/Sun after 2),18-Hole Round of Golf w/ Cart for 2 (Monday-Friday anytime, Sat/Sun after 2pm),9- Hole Round of Golf with Cart for 2 (Monday-Friday),18-Hole Round of Golf w/ Carts for 4 (Monday-Friday Anytime)",3
14533,"Golf. Relax at Chosen Valley Golf Club with walking golf for groups and individuals, plus enjoy up to 42% off.,Relax at Chosen Valley Golf Club with walking golf for groups and individuals, plus enjoy up to 42% off.,Relax at Chosen Valley Golf Club with walking golf for groups and individuals, plus enjoy up to 42% off.. 9-Hole Round of Golf for Four,9-Hole Round of Golf for Two,9-Hole Round of Golf for One",3
17002,"Golf. Discover Golf: Five-Week Learn How to Play Golf Class for One or Two at Cypresswood Golf Club (Up to 84% Off),Discover Golf: Five-Week Learn How to Play Golf Class for One or Two at Cypresswood Golf Club (Up to 84% Off). Five-Week ""Learn To Play Golf"" Class for One,Five-Week ""Learn To Play Golf"" Class for Two",3
17319,"Golf. Up to 28% Off on Golf at DAS Golf Lessons,Up to 28% Off on Golf at DAS Golf Lessons,Up to 28% Off on Golf at DAS Golf Lessons,Up to 28% Off on Golf at DAS Golf Lessons. One - 60 Minute Golf Lesson with with a copy of ""The Method: A Golf Success Strategy ”,Three - 60 Minute Golf Lessons with with a copy of “Your Yardage Book”,Two - 60 minute Golf Lessons with with a copy of “Your Yardage Book”,One - 60 minute Golf Lesson with with a copy of “Your Yardage Book”",3
21418,"Golf. Up to 44% Off on Indoor Golf at Envision Golf,Up to 44% Off on Indoor Golf at Envision Golf. One hour Standard Bay Golf Rental for up to 6 people, Valid to,Two hour Standard Bay Golf Rental for up to 6 people, Valid to",3


In [42]:
df[['text', 'cluster']].query("cluster == 43").head(10)

,text,cluster
3584,Movies. AMC 19th St. East 6. Tickets for Select Showtimes,43
3585,Movies. AMC 309 Cinema 9. Tickets for Select Showtimes,43
3586,Movies. AMC 34th Street 14. Tickets for Select Showtimes,43
3587,Movies. AMC 84th Street 6. Tickets for Select Showtimes,43
3588,Movies. AMC 9+CO 10. Tickets for Select Showtimes,43
3589,Movies. AMC Academy 8. Tickets for Select Showtimes,43
3590,Movies. AMC Ahwatukee 24. Tickets for Select Showtimes,43
3591,Movies. AMC Alderwood Mall 16. Tickets for Select Showtimes,43
3592,Movies. AMC Allegany 8. Tickets for Select Showtimes,43
3593,Movies. AMC Altamonte Mall 18. Tickets for Select Showtimes,43


In [43]:
df[['text', 'cluster']].query("cluster == 44").head(10)

,text,cluster
334,Botox. $169 for 20 Units of Botox at 4Ever Young Aventura ($205 Value). 20 Units of Botox for One Area,44
351,Botox. 20 Units of Botox for One Area at 4Ever Young New Braunfels (Up to 33% Off). 20 Units of Botox for One Area,44
358,Botox. 20 Units of Botox for One Area at 4Ever Young Orlando Lake Ivanhoe ( Up to 33% Off ). 20 Units of Botox for One Area,44
362,Botox. 20 Units of Botox for One Area from 4Ever Young St Johns (Up to 33% Off). 20 Units of Botox for One Area,44
421,Botox. Transform Your Looks as You Discover the Ultimate Botox Experience at 4Ever Young Apex (Up to 33% Off). 20 Units of Botox for One Area,44
425,Botox. Secret to Timeless Beauty at 4Ever Young Boca Raton: 20 Units of Botox for One Area (Up To 30% Off). 20 Units of Botox for One Area,44
430,Botox. Discover the Rejuvenation at 4Ever Young Bridgeport | 20 Units of Botox Injection for One Area (Up To 33% Off). 20 Units of Botox for One Area,44
439,Botox. Turn Back Time with 4Ever Young—Save up to 25% on 20 Units of Botox for One Area. 20 Units of Botox for One Area,44
450,Botox. Secret to Timeless Beauty at 4Ever Young Falls Church: 20 Units of Botox for One Area . 20 Units of Botox for One Area,44
456,Botox. Discover the Ultimate Rejuvenation with 4Ever Young Fort Lauderdale: 20 Units of Botox for One Area. 20 Units of Botox for One Area,44


In [50]:
df[['text', 'cluster']].query("cluster == 51").head(10)

,text,cluster
723,"Oil Change. Enjoy up to 46% off on A-1 Quality Car Care's full synthetic oil change with a 29-point inspection in Palm Springs,Enjoy up to 46% off on A-1 Quality Car Care's full synthetic oil change with a 29-point inspection in Palm Springs,Enjoy up to 46% off on A-1 Quality Car Care's full synthetic oil change with a 29-point inspection in Palm Springs. Three Full Synthetic Oil Change with 29-Point Inspection,One Full Synthetic Oil Change with 29-Point Inspection,Two Full Synthetic Oil Change with 29-Point Inspection",51
729,"Oil Change. A & A-OIL & WASH OPERATIONS INC offers full synthetic oil change with filter replacement, up to 49% off. $9.99 off Full Synthetic Oil Change with Filter Replacement",51
1061,Oil Change. Up to 10% Off on Oil Change at A One Auto Body And Repair. Synthetic Oil Change,51
1131,"Oil Change. Experience A And S Auto Clinic's Oil Change Options with 20-Point Inspection for up to 41% Off,Experience A And S Auto Clinic's Oil Change Options with 20-Point Inspection for up to 41% Off. 8033 Snouffer School Road: Semi-Synthetic Oil Change with 20-Point Inspection,8033 Snouffer School Road: Synthetic Oil Change with 20-Point Inspection",51
1200,"Oil Change. Semi- or Full Synthetic Oil Change with Tire Rotation and Brake Inspection at A1 Automotive (Up To 63% Off) ,Semi- or Full Synthetic Oil Change with Tire Rotation and Brake Inspection at A1 Automotive (Up To 63% Off) . Full Synthetic Oil Change with Tire Rotation and Brake Inspection,Semi-Synthetic Oil Change with Tire Rotation and Brake Inspection",51
1251,"Oil Change. Experience top-notch oil change options at AAA Tires & Lube Co., offering up to 43% off for full-synthetic, high mileage, and semi-synthetic oils.,Experience top-notch oil change options at AAA Tires & Lube Co., offering up to 43% off for full-synthetic, high mileage, and semi-synthetic oils.,Experience top-notch oil change options at AAA Tires & Lube Co., offering up to 43% off for full-synthetic, high mileage, and semi-synthetic oils.. Semi-Synthetic Oil Change,High-Mileage Semi-Synthetic Oil Change,Full Synthetic Oil Change",51
1264,"Oil Change. Diesel Oil Change for Cars, SUV's, and Trucks or Bring Your Own Oil to Change at AAG Auto Repair (Up to 46% Off),Diesel Oil Change for Cars, SUV's, and Trucks or Bring Your Own Oil to Change at AAG Auto Repair (Up to 46% Off),Diesel Oil Change for Cars, SUV's, and Trucks or Bring Your Own Oil to Change at AAG Auto Repair (Up to 46% Off),Diesel Oil Change for Cars, SUV's, and Trucks or Bring Your Own Oil to Change at AAG Auto Repair (Up to 46% Off). One Diesel Engine Oil Change for Truck (Rotella or House Brand) Up to 12 Quarts of 10w-30 or 15w-40 Oil & Filter,Diesel Oil Change For SUV - 6 Quarts 10w-30 or 15w-40 Oil & Filter,Diesel Oil Change For Car 5 quarts 0f 10w-30 0r 15w-40 Oil & Filter,Bring your Own Diesel Oil & Filter and We Change",51
1281,"Oil Change. Up to 36% Off on Oil Change at AAG Auto Repair,Up to 36% Off on Oil Change at AAG Auto Repair,Up to 36% Off on Oil Change at AAG Auto Repair,Up to 36% Off on Oil Change at AAG Auto Repair,Up to 36% Off on Oil Change at AAG Auto Repair,Up to 36% Off on Oil Change at AAG Auto Repair. Excessive Miles Full Synthetic Oil Change With Filter,3349 183rd Street: Semi-Synthetic Oil Change,3349 183rd Street: Conventional Oil Change,Excessive Miles Conventional Oil Change With Filter,Excessive Miles Synthetic Blend Oil Change With Filter,3349 183rd Street: Synthetic Oil Change",51
1286,"Oil Change. Oil Change at AAMCO Transmissions and Total Car Care (Up to 54% Off). Two Options Available.,Oil Change at AAMCO Transmissions and Total Car Care (Up to 54% Off). Two Options Available.. 401 West High Street: Semi-Synthetic Oil Change with Free Multi-Point Inspection,401 West High Street: Synthetic Oil Change with Free Multi-Point Inspection",51
1311,"Oil Change. Up to 88% Off on Oil Change - Full Service at ABAS AUTO REPAIR,Up to 88% Off on Oil Change - Full Service at ABAS AUTO REPAI

In [51]:
df[['text', 'cluster']].query("cluster == 52").head(10)

,text,cluster
56,"Repair Services. Enhance your vehicle's safety with 106 St. Tire & Wheel's brake pad replacement options, up to 35% off.,Enhance your vehicle's safety with 106 St. Tire & Wheel's brake pad replacement options, up to 35% off.. 2 sets of Brake Pad Replacement including a 12 month warranty,One Set of Front or rear BRAKE PADS Replacement",52
126,"Repair Services. Up to 35% Off on Brake Pad Replacement at 2'u'Brakes,Up to 35% Off on Brake Pad Replacement at 2'u'Brakes. Mobile Brake Pad Replacement for Front and Rear Pads/Gold,Mobile Brake Pad Replacement for Front or Rear Pads/Gold",52
134,"Repair Services. Up to 57% Off on Automotive Service / Repair at 21Auto,Up to 57% Off on Automotive Service / Repair at 21Auto. Brake Pads Replacement - Front & Rear,Brake Pads Replacement",52
156,"Repair Services. At 24 Hour Alignment Express, enjoy comprehensive brake services with options for front, rear or both at up to 53% off,At 24 Hour Alignment Express, enjoy comprehensive brake services with options for front, rear or both at up to 53% off. FRONT AND REAR BRAKES REPLACEMENT,FRONT OR REAR BRAKES",52
157,"Repair Services. Up to 52% Off on Automotive Service / Repair at 24 hour alignment express,Up to 52% Off on Automotive Service / Repair at 24 hour alignment express. REAR AND FRONT SHOCK OR STRUT REPLACEMENT NOT INCLUDING PARTS LABOR ONLY,SHOCKS AND STRUT REPLACEMENT REAR OR FRONT LABOR ONLY PARTS NOT INCLUDED",52
158,Tires & Wheels. $49 for Four-Wheel Alignment at 24 Express Alignment ($89.95 Value). 1403 South Loop West: Four-Wheel Alignment,52
212,"Repair Services. Ensure vehicle safety with 2nd Chance auto services' mobile brake pad replacement and oil change, up to 50% off,Ensure vehicle safety with 2nd Chance auto services' mobile brake pad replacement and oil change, up to 50% off,Ensure vehicle safety with 2nd Chance auto services' mobile brake pad replacement and oil change, up to 50% off. Basic Mobile Front & back brake pad replacement & oil change,Elite Mobile Front & Back brake pad replacement & Oil Change,Premium Mobile front & back brake pad replacement & oil change",52
725,Tires & Wheels. Up to 29% Off on Tire Rotation at A & A Auto LLC. Tire Rotation,52
728,"Repair Services. A & A-OIL & WASH OPERATIONS INC offers brake pad replacement with car wash service, up to 66% off. *$50 Off* One Set of Front/rear BRAKE PADS Replacement",52
1060,Tires & Wheels. Up to 10% Off on Wheel Alignment / Balancing - Car at A One Auto Body And Repair. All-Wheel Computerized Alignment,52


In [53]:
df[['text', 'cluster']].query("cluster == 54").head(10)

,text,cluster
376,"Cosmetic Procedures. Achieve stunning curves at 4 See LLC with non-invasive butt lifts and enjoy up to 48% off tailored treatments,Achieve stunning curves at 4 See LLC with non-invasive butt lifts and enjoy up to 48% off tailored treatments,Achieve stunning curves at 4 See LLC with non-invasive butt lifts and enjoy up to 48% off tailored treatments. Two Non-Invasive ThermaLift Butt-Lift Sessions,Six Non-Invasive ThermaLift Butt-Lift Sessions,Four Non-Invasive ThermaLift Butt-Lift Sessions",54
378,"Non-Surgical Facelifts. 4 See Llc offers Ultra-Lift Skin Tightening Sessions to refresh your look, with up to 47% off for ultimate skincare benefits.,4 See Llc offers Ultra-Lift Skin Tightening Sessions to refresh your look, with up to 47% off for ultimate skincare benefits.,4 See Llc offers Ultra-Lift Skin Tightening Sessions to refresh your look, with up to 47% off for ultimate skincare benefits.. Two Ultra-Lift Skin-Tightening Sessions,Three Ultra-Lift Skin-Tightening Sessions,One Ultra-Lift Skin-Tightening Session",54
380,"Breast Augmentation. Discover non-surgical breast lifts with 4 See Llc, from 2 to 6 sessions, offering up to 48% off for youthful results,Discover non-surgical breast lifts with 4 See Llc, from 2 to 6 sessions, offering up to 48% off for youthful results,Discover non-surgical breast lifts with 4 See Llc, from 2 to 6 sessions, offering up to 48% off for youthful results. Four Non-Invasive ThermaLift Breast Lifts,Six Non-Invasive ThermaLift Breast Lifts,Two Non-Invasive ThermaLift Breast Lifts",54
681,"Cosmetic Procedures. Up to 50% Off on Gluteoplasty / Butt Lift at Haute & Bodied Beauty Lounge,Up to 50% Off on Gluteoplasty / Butt Lift at Haute & Bodied Beauty Lounge,Up to 50% Off on Gluteoplasty / Butt Lift at Haute & Bodied Beauty Lounge. Three Vacuum Butt-Lift Treatments,One Vacuum Butt-Lift Treatment,Six Vacuum Butt-Lift Treatments",54
686,"Cosmetic Procedures. Achieve fuller curves with Haute & Bodied Beauty Lounge's vacuum butt-lift sessions offering up to 35% off,Achieve fuller curves with Haute & Bodied Beauty Lounge's vacuum butt-lift sessions offering up to 35% off,Achieve fuller curves with Haute & Bodied Beauty Lounge's vacuum butt-lift sessions offering up to 35% off. One 40-Minute Noninvasive Vacuum Butt-Lift Session,Three 40-Minute Noninvasive Vacuum Butt-Lift Sessions,Six 40-Minute Noninvasive Vacuum Butt-Lift Sessions",54
741,"Cosmetic Procedures. A&N Beauty Bar's non-surgical butt lift offers up to 39% off for a beautiful transformation.,A&N Beauty Bar's non-surgical butt lift offers up to 39% off for a beautiful transformation.. Brazilian Butt Lifting,Two Brazilian Butt Lifting Sessions",54
1015,"Non-Surgical Facelifts. One, Two, or Three Ultra-Lift Skin Tightening Sessions at A New Slimmer You (Up to 74% Off),One, Two, or Three Ultra-Lift Skin Tightening Sessions at A New Slimmer You (Up to 74% Off),One, Two, or Three Ultra-Lift Skin Tightening Sessions at A New Slimmer You (Up to 74% Off). Three Ultra-Lift Skin Tightening Sessions,One Ultra-Lift Skin Tightening Session,Two Ultra-Lift Skin Tightening Sessions",54
1018,"Cosmetic Procedures. Two, Four, or Six Therma Lift Butt Lift Treatments at A New Slimmer You (Up to 80% Off),Two, Four, or Six Therma Lift Butt Lift Treatments at A New Slimmer You (Up to 80% Off),Two, Four, or Six Therma Lift Butt Lift Treatments at A New Slimmer You (Up to 80% Off). Six Therma Lift Butt Lift Treatments,Four Therma Lift Butt Lift Treatments,Two Therma Lift Butt Lift Treatments",54
1020,"Facelift. Discover A New Slimmer You: Hendersonville with Therma-Lift Skin Tightening Facelifts up to 78%,Discover A New Slimmer You: Hendersonville with Therma-Lift Skin Tightening Facelifts up to 78%,Discover A New Slimmer You: Hendersonville with Therma-Lift Skin Tightening Facelifts up to 78%. Two Therma Lift Skin-Tightening Facelifts,One Therma Lift Skin-Tightening Facelift,Four Therma Lift Skin-Tightening Facelifts",54
1021,"Cosmetic Procedures. Experie

In [54]:
df[['text', 'cluster']].query("cluster == 55").head(10)

,text,cluster
81,"Laser Hair Removal. Six Laser Hair Removal Sessions on One Body Area for Silky Smooth Skin at 128 Luxury Health Spa - Save up to 62%,Six Laser Hair Removal Sessions on One Body Area for Silky Smooth Skin at 128 Luxury Health Spa - Save up to 62%,Six Laser Hair Removal Sessions on One Body Area for Silky Smooth Skin at 128 Luxury Health Spa - Save up to 62%,Six Laser Hair Removal Sessions on One Body Area for Silky Smooth Skin at 128 Luxury Health Spa - Save up to 62%,Six Laser Hair Removal Sessions on One Body Area for Silky Smooth Skin at 128 Luxury Health Spa - Save up to 62%. Six Diolaze Hair Removal Sessions - Bikini / Stomach / Chest,Six Diolaze Hair Removal Sessions - Arms / Legs,Six Diolaze Hair Removal Sessions - Chin / Upper Lip / Side Burns,Six Diolaze Hair Removal Sessions - Back,Six Diolaze Hair Removal Sessions - Neck / Underarms",55
151,"Tattoo Removal. Experience MD Cosmetic Clinic's laser tattoo removal sessions for small to medium areas, with savings up to 30%,Experience MD Cosmetic Clinic's laser tattoo removal sessions for small to medium areas, with savings up to 30%. Three Laser Tattoo Removal Sessions on a Small Area,Three Laser Tattoo Removal Sessions on a Medium Area",55
168,"Laser Hair Removal. Experience up to 74% off laser hair removal options at 247 Natural Wellness Center for a smoother, hair-free you,Experience up to 74% off laser hair removal options at 247 Natural Wellness Center for a smoother, hair-free you,Experience up to 74% off laser hair removal options at 247 Natural Wellness Center for a smoother, hair-free you,Experience up to 74% off laser hair removal options at 247 Natural Wellness Center for a smoother, hair-free you. One Laser Hair-Removal Session for Full Brazilian with Bikini Line and Inner Buttock Included (Trial),One Laser Hair-Removal Session for Full Manzilian with Bikini Line and Inner Buttock Included (Trial),One Laser Hair-Removal Session for the Entire Body with Face Included (Trial),One Laser Hair-Removal Session for Full Brazilian and Armpits (Trial)",55
169,"Laser Hair Removal. Experience up to 90% off at 247 Natural Wellness Center with Unlimited Laser Hair Removal Sessions on Varied Areas,Experience up to 90% off at 247 Natural Wellness Center with Unlimited Laser Hair Removal Sessions on Varied Areas,Experience up to 90% off at 247 Natural Wellness Center with Unlimited Laser Hair Removal Sessions on Varied Areas,Experience up to 90% off at 247 Natural Wellness Center with Unlimited Laser Hair Removal Sessions on Varied Areas. One Year of Unlimited Laser Hair-Removal Sessions on One Small and One Medium Areas,One Year of Unlimited Laser Hair-Removal Sessions on One Small, Medium, and Large Area,One Year of Unlimited Laser Hair-Removal Sessions for Full Body,One Year of Unlimited Laser Hair-Removal Sessions on One Small Area",55
172,"Laser Hair Removal. Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off,Discover 247 Natural Wellness Center's tailored laser hair removal services for all body areas, with up to 91% off. Three Laser Hair-Removal Sessions on One Extra Small Area,Six Laser Hair-Removal Sessions on One Large Area,Six Laser Hair-Removal Sessions on One Medium Area,Three Laser Hair-Removal Sessions on

In [68]:
df[['text', 'cluster']].query("cluster == 68").head(10)

,text,cluster
112,"Dance Classes. Up to 78% Off on Dance Class at 1st Dance Chicago,Up to 78% Off on Dance Class at 1st Dance Chicago,Up to 78% Off on Dance Class at 1st Dance Chicago,Up to 78% Off on Dance Class at 1st Dance Chicago. Wedding Dance Consultation,6 Lesson Wedding Package,6 Social Dance Classes,Introductory Sampler",68
231,"Dance Classes. Dance Your Heart Out at 305 Fitness: Join the Party with 3 Dance Fitness Classes, up to 33% off. Three 305 Fitness Classes",68
555,"Fitness Classes. Up to 49% Off on Chair and Belly Dance - Online Classes at 4play fitness,Up to 49% Off on Chair and Belly Dance - Online Classes at 4play fitness. One Month of Online Classes - Including Chair Dance & Belly Dance,6 Months of Online Classes - Including Chair Dance & Belly Dance",68
1148,Dance Classes. Discover Master the Elements of Social Dance with A Step to Gold and save up to 50% on this transformative online experience. Master the Elements of Social Dance,68
1149,Dance Classes. Get a Wedding First Dance Consultation for Two at A Step to Gold International Ballroom (Up to 73% Off). Wedding First Dance Consultation for 2,68
1238,"Dance Classes. Up to 55% Off on Dance Class at AA Studio,Up to 55% Off on Dance Class at AA Studio,Up to 55% Off on Dance Class at AA Studio. One Drop-In Dance Class for One Adult (Ages 18 and Up),One Private Hip Hop Dance Class,One Month of Unlimited Drop-In Dance Classes for One Adult (Ages 18 and Up)",68
1294,Dance Classes. Up to 50% Off on Salsa Dancing Class at AATMA Dance Studio. Five Beginner Salsa Classes for One,68
1305,"Dance Classes. Up to 65% Off on Dancing at AB Dance Schools,Up to 65% Off on Dancing at AB Dance Schools. Introductory Offer: Two 20-Minute Private Dance Lessons,Wedding Dance Consultation: One 30-Minute Private Lesson",68
1318,"Dance Classes. Discover Diverse Dance Styles at Abayas' Ballroom with Private Lessons Up to 32% Off,Discover Diverse Dance Styles at Abayas' Ballroom with Private Lessons Up to 32% Off,Discover Diverse Dance Styles at Abayas' Ballroom with Private Lessons Up to 32% Off. Two 45-min Private Lessons for Two,Two 45-min Private Lessons for One,Two 45-Minute Wedding Lessons for Two",68
1410,"Dance Classes. Explore social dancing with Absolute Dance Studio offering salsa, swing, ballroom classes and more, up to 50% off. 3 Private dance lessons, One group less and one practice session",68


In [75]:
df[['text', 'cluster']].query("cluster == 75").head(10)

,text,cluster
298,"No-Chip Mani & Reg Pedi. Up to 43% Off on Salon-Shellac/No-Chip Mani-Pedi at 3D Lash And Brow,Up to 43% Off on Salon-Shellac/No-Chip Mani-Pedi at 3D Lash And Brow,Up to 43% Off on Salon-Shellac/No-Chip Mani-Pedi at 3D Lash And Brow. One Spa Pedicure,One Gel Manicure,One Gel Manicure and Spa Pedicure",75
307,"No-Chip Mani & Reg Pedi. Discover 3D Lash and Brow Salon's Acrylic, Dip, or Gel Manicures up to 26% Off,Discover 3D Lash and Brow Salon's Acrylic, Dip, or Gel Manicures up to 26% Off,Discover 3D Lash and Brow Salon's Acrylic, Dip, or Gel Manicures up to 26% Off. Gel Manicure,Dip Manicure,Acrylic Full Set",75
324,"Manicure. Experience 4 Beauty Spa's Gel Manicure with a nail trim, buff, shaping, and a hand massage, up to 31% off. Gel Manicure",75
582,"Mani Pedi. Regal 5 STARS HAIR & NAILS - MILPITAS: Fill-in or Full Acrylic Nails and Luxe Pedicures (Up to 42% Off),Regal 5 STARS HAIR & NAILS - MILPITAS: Fill-in or Full Acrylic Nails and Luxe Pedicures (Up to 42% Off). Gel Mani-Pedi,Full Set Acrylic 2 Nail Design",75
613,"Pedicure. Discover the ultimate nail experience at 5C Blink with Gel Manicure, Pedicure, SNS Manicure & Brow Waxing, up to 9% off,Discover the ultimate nail experience at 5C Blink with Gel Manicure, Pedicure, SNS Manicure & Brow Waxing, up to 9% off. Gel Manicure and One Classic Pedicure,SNS Powder Manicure with Eyebrow Waxing",75
619,"No-Chip Manicure. Transform Your Nails with Manicure, Pedicure and Add On Treatment at 5T Beauty Academy (Up to 52% Off),Transform Your Nails with Manicure, Pedicure and Add On Treatment at 5T Beauty Academy (Up to 52% Off). One Classic Manicure with Add On Treatment,Two Classic Manicure with Pedicure Add On Treatment",75
680,"Nail Designs. Dripped Tipz offers nail art services with tailored care, including manicures and pedicures up to 35% off,Dripped Tipz offers nail art services with tailored care, including manicures and pedicures up to 35% off. One Shellac/ Gel Manicure,Acrylic Full Set",75
707,"Manicure. Up to 40% Off on Nail Salon - Manicure at 99 Beauty Salon&School,Up to 40% Off on Nail Salon - Manicure at 99 Beauty Salon&School,Up to 40% Off on Nail Salon - Manicure at 99 Beauty Salon&School,Up to 40% Off on Nail Salon - Manicure at 99 Beauty Salon&School. Acrylic full set with gel,Gel manicure,Acrylic fill with gel,Dip powder manicure",75
736,"No-Chip Mani & Reg Pedi. Transform your look at A & A Nails Company LLC with basic to supreme mani-pedis, offering up to 10% off,Transform your look at A & A Nails Company LLC with basic to supreme mani-pedis, offering up to 10% off,Transform your look at A & A Nails Company LLC with basic to supreme mani-pedis, offering up to 10% off,Transform your look at A & A Nails Company LLC with basic to supreme mani-pedis, offering up to 10% off,Transform your look at A & A Nails Company LLC with basic to supreme mani-pedis, offering up to 10% off,Transform your look at A & A Nails Company LLC with basic to supreme mani-pedis, offering up to 10% off. Regular Manicure and Deluxe Pedicure,Supreme Pedicure with Cucumber Mask and Paraffin Wax,Regular Pedicure with Shellac,Deluxe Manicure and Deluxe Pedicure,Supreme Pedicure,Regular Manicure and Supreme Pedicure",75
870,"Manicure. Up to 30% Off on Nail Salon - Manicure at A Luxea Nailspa,Up to 30% Off on Nail Salon - Manicure at A Luxea Nailspa,Up to 30% Off on Nail Salon - Manicure at A Luxea Nailspa,Up to 30% Off on Nail Salon - Manicure at A Luxea Nailspa. Dipping full set and basic gel pedicure ,Special pedicure,Manicure gel and basic pedicure gel,GelX fullset",75


In [79]:
df[['text', 'cluster']].query("cluster == 80").head(10)

,text,cluster
164,"Phone Repair. Up to 46% Off on Mobile Phone / Smartphone Repair at 247 iPhone Repair,Up to 46% Off on Mobile Phone / Smartphone Repair at 247 iPhone Repair,Up to 46% Off on Mobile Phone / Smartphone Repair at 247 iPhone Repair,Up to 46% Off on Mobile Phone / Smartphone Repair at 247 iPhone Repair,Up to 46% Off on Mobile Phone / Smartphone Repair at 247 iPhone Repair,Up to 46% Off on Mobile Phone / Smartphone Repair at 247 iPhone Repair. iPhone 6,7,8 and plus models,iPhone XS Max Glass and LCD Screen Repair,iPhone XS Glass and LCD Screen Repair,iPhone Customer Diagnostic,iPhone 11 Pro or 11 Pro Max Glass and LCD Screen Repair,iPhone X or Xr Glass and LCD Screen Repair",80
165,"Electronics. Up to 34% Off on Cell Phone / Smartphone Store at 247 iPhone Repair,Up to 34% Off on Cell Phone / Smartphone Store at 247 iPhone Repair,Up to 34% Off on Cell Phone / Smartphone Store at 247 iPhone Repair. iPhone 6/7/8 & Plus Models Battery Replacement,iPhone X/XR/XS/ XS Max Battery Replacement,iPhone 11/12/13/14 Battery Replacement",80
166,"Electronics Repair. Up to 30% Off on Personal Electronics Repair at 247 iPhone Repair,Up to 30% Off on Personal Electronics Repair at 247 iPhone Repair,Up to 30% Off on Personal Electronics Repair at 247 iPhone Repair. MacBook Air 13 (A1932, Mid 2019/A2179, Early 2020) Complete LCD Display Assembly Replacement,MacBook Pro Touchbar 13 (A1706, Late 2016, Mid 2017) Complete LCD Display Assembly Replacement,MacBook Air 13 (A2337, Late 2020) Complete LCD Display Assembly Replacement",80
1220,"Phone Repair. Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off,Experience fast on-location repairs with A2Z Cellphone Repairs, from iPad glass fixes to iPhone LCD replacements, up to 20% off. i phone 13 pro max lcd screen reaplacement,I PAD 5/6/7/8/9 GENERATION DIGITIZERGLASS REPAIR,iPhone 6, 6S, 6 Plus, 6S Plus, 7, 8 Glass and LCD Screen Repair,I PHONE XSMAX/11 PRO /11 PRO MAX GLASS AND LCD SCREEN REPAIR,iPhone 7 plus or 8 Plus Glass and LCD Screen Repair,iPhone X /XS/XR/11 Glass and LCD Screen Repair,iPhone 12 Glass and LCD Screen Repair,i phone 13 /14 lcd screen replacement",80
1776,"Electronics. Experience ADR Squad's expert iPhone back glass repairs from models 8 to 14 Pro Max with up to 56% savings,Experience ADR Squad's expert iPhone back glass repairs from models 8 to 14 Pro Max with up to 56% savings,Experience ADR Squad's expert iPhone back glass repairs from models 8 to 14 Pro Max with up to 56% savings,Experience ADR Squad's expert iPhone back glass repairs from models 8 to 14 Pro Max with up to 56% savings,Experience ADR Squad's expert iPhone back glass repairs from models 8 to 14 Pro Max with up to 56% savings,Experience ADR Squad's expert iPhone back glass repairs from models 8 to 14 Pro Max with up to 56% savings. iPhone 14 Pro or 14 Pro Max Back Glass Repair,iPhone 13 Pro or 13 Pro Max Back Glass Repair,iPhone 11, 11 Pro, or 11 Pro Max Back Glass Repair,iPhone 8, 8 Plus, or iPhone X Back Glass Repair,iPhone XR, XS, or XS Max Back Glass Repair,iPhone 13 mini,13 Back Glass Repair",80
1948,"Phone Repair. Advanced Repair Center offers reliable iPhone screen rep

In [98]:
df[['text', 'cluster']].query("cluster == 99").head(10)

,text,cluster
59961,Asian Restaurants. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99
59965,American Restaurants. $20 for $40 Worth of Casual Dining. $20 for $40 Worth of Casual Dining,99
59969,Steakhouse. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99
59972,Mediterranean Restaurants. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99
59979,American Restaurants. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99
59981,American Restaurants. $10 For $20 Worth Of Casual Dining. $10 For $20 Worth Of Casual Dining,99
59983,Steakhouse. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99
59990,French Restaurants. $10 For $20 Worth Of Cafe Dining & More. $10 For $20 Worth Of Cafe Dining & More,99
59991,American Restaurants. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99
59993,American Restaurants. $15 For $30 Worth Of Casual Dining. $15 For $30 Worth Of Casual Dining,99


In [123]:
number_of_clusters = 10
kmeans1 = KMeans(n_clusters=number_of_clusters)
kmeans1.fit(tfidf_matrix)
df['cluster1'] = kmeans1.labels_

In [133]:
df[['text', 'cluster1']].query("cluster1 == 9").head(10)

,text,cluster1
4701,"Facial. Pamper Your Back with One or Two 60-Minute Back Facials at Anointed SkinKare (Up to 37% Off),Pamper Your Back with One or Two 60-Minute Back Facials at Anointed SkinKare (Up to 37% Off). Two 60-Minute Back Facials,One 60-Minute Back Facial",9
12420,"Windshield & Windows. Novus Glass Phoenix offers mobile windshield replacement with up to 90% off and up to $150 cash back benefits,Novus Glass Phoenix offers mobile windshield replacement with up to 90% off and up to $150 cash back benefits. $150 Cash Back on Mobile Windshield Replacement with Insurance,$100 Cash Back on Mobile Windshield Replacement without Insurance",9
12923,Facial. Indulge in a Deep Cleansing Back Treatment at Monet Esthetics with up to 40% off. Back Treatment,9
14494,"Spas. Chocolate Box Skin offers refreshing back scrub experience, revealing brighter skin with up to 55% off. Back scrub",9
30105,"Hair Extensions & Wigs. Back Decompression Belt Lumbar Support Lower Back Traction Device 1-2 Pack,Back Decompression Belt Lumbar Support Lower Back Traction Device 1-2 Pack. Back Decompression Belt Lumbar Support Lower Back Traction Device 1-2 Pack Beige 2 Packs,Back Decompression Belt Lumbar Support Lower Back Traction Device 1-2 Pack Beige 1 Pack",9
30106,Hair Extensions & Wigs. Back Decompression Belt Lumbar Traction Device for Back Pain Relief Great Gift . Back Decompression Belt Lumbar Traction Device for Back Pain Relief Great Gift Beige 10*5*3 in,9
75212,"Windshield & Windows. Experience ProLite Auto Glass with up to 92% off on in-shop or mobile windshield services and enjoy cash back,Experience ProLite Auto Glass with up to 92% off on in-shop or mobile windshield services and enjoy cash back,Experience ProLite Auto Glass with up to 92% off on in-shop or mobile windshield services and enjoy cash back,Experience ProLite Auto Glass with up to 92% off on in-shop or mobile windshield services and enjoy cash back. $100 Towards In-Shop Windshield Replacement; Valid with Cash Only,$150 Cash Back on Windshield Replacement with Insurance (In Shop),$125 Cash Back on Mobile Windshield Replacement with Insurance,In-Shop Windshield Chip Repair for Up to Two Quarter-Size Chips",9
75549,"Facial. Revitalize your Back with the Pure Beauty Back Treatment at Pure Beauty by Sierra (Up to 42% Off),Revitalize your Back with the Pure Beauty Back Treatment at Pure Beauty by Sierra (Up to 42% Off). Pure Beauty Back Treatment,Mini Back Treatment",9
77831,Steakhouse. 5% Cash Back at 101 Steak. 5% Cash Back at 101 Steak,9
77832,American Restaurants. 5% Cash Back at 107 State. 5% Cash Back at 107 State,9
